# A Recall-Oriented Operating Point: the F2 Threshold

The selected Step 2 SVM misses about a quarter of the fake postings at the F1 threshold (recall 0.752). For a platform, a missed fake posting usually costs more than a false alarm: the job seeker can lose money or personal data, while a false alarm only sends a real posting to a reviewer. This notebook adds a second decision threshold that weights recall more, without changing the model.

**F2 score.** F2 = 5·P·R / (4·P + R) counts recall twice as much as precision (F-beta with beta = 2).

**Method.** Inside every outer fold, the F1 threshold is chosen on the inner out-of-fold scores of the training fold, as before. The F2 threshold is chosen on the **same** inner out-of-fold scores by maximising F2 (`best_fbeta_threshold`). Both thresholds are applied to the same outer-fold scores, so the comparison is paired and the outer fold is never used to choose a threshold. The model, its scores and therefore PR-AUC are unchanged. The test set is not used.

### Decision rule (fixed before running)

The F2 threshold is kept as a second operating point ("review mode") when, compared with the F1 threshold on the same scores, its **mean paired F2 gain is positive and it is better in at least 12 of the 15 folds**. F1 stays the primary metric of the project; the F2 threshold is reported next to it, not instead of it.

In [1]:
import sklearn

if sklearn.__version__ != "1.6.1":
    raise RuntimeError(
        f"This notebook uses scikit-learn 1.6.1 (found {sklearn.__version__}). "
        "Install the pinned versions with: pip install -r requirements.txt"
    )

import time
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

from pathlib import Path

import numpy as np
import pandas as pd

from fakejob.data import load_split
from fakejob.evaluation import repeated_group_cv
from fakejob.features import STEP2_CONFIG, build_pipeline, posting_features

pd.set_option("display.precision", 3)
RESULTS_DIR = Path("results")
METRICS = ["average_precision", "roc_auc", "f1", "precision", "recall"]

split = load_split()
features = posting_features(split.model_df)
dev_index = split.dev_index
X_dev = features.loc[dev_index]
y_dev = split.y.loc[dev_index]
groups_dev = split.groups.loc[dev_index]
print(f"Development data: {len(dev_index)} postings, {int(y_dev.sum())} fake "
      f"({100 * y_dev.mean():.2f}%)")

Development data: 15001 postings, 724 fake (4.83%)


## Cross-Validation

The assertion checks that the F1-threshold results are identical to the saved Step 2 results, so adding the F2 threshold did not change anything else.

In [2]:
start = time.time()
result = repeated_group_cv(
    build_pipeline(STEP2_CONFIG), X_dev, y_dev, groups_dev, name="Step 2 SVM",
    n_splits=5, n_repeats=3, inner_splits=3, seed=0, n_jobs=4, fbeta=2,
)
folds = result.folds
folds.to_csv(RESULTS_DIR / "f2_threshold_folds.csv", index=False)
print(f"{time.time() - start:.0f}s")

saved_step2 = pd.read_csv(RESULTS_DIR / "step2_combined_folds.csv")
assert np.allclose(folds[METRICS + ["threshold"]].values, saved_step2[METRICS + ["threshold"]].values)
print("F1-threshold results identical to the saved Step 2 results.")

700s
F1-threshold results identical to the saved Step 2 results.


## F1 Threshold vs F2 Threshold

Mean ± standard deviation over the 15 outer folds, and the paired difference (F2 threshold minus F1 threshold) on the same scores.

In [3]:
modes = {
    "F1 threshold": {"threshold": "threshold", "precision": "precision", "recall": "recall",
                     "F1": "f1", "F2": "f2_at_f1_threshold"},
    "F2 threshold": {"threshold": "threshold_f2", "precision": "precision_f2", "recall": "recall_f2",
                     "F1": "f1_f2", "F2": "f2"},
}
summary = pd.DataFrame({
    mode: {metric: f"{folds[col].mean():.3f} ± {folds[col].std():.3f}" for metric, col in cols.items()}
    for mode, cols in modes.items()
}).T
display(summary)

rows = []
for metric in ["precision", "recall", "F1", "F2"]:
    diff = folds[modes["F2 threshold"][metric]] - folds[modes["F1 threshold"][metric]]
    rows.append({"metric": metric, "mean difference": diff.mean(), "std": diff.std(),
                 "folds higher": f"{int((diff > 0).sum())}/{len(diff)}"})
paired = pd.DataFrame(rows).set_index("metric")
display(paired)

f2_diff = folds["f2"] - folds["f2_at_f1_threshold"]
passes = bool(f2_diff.mean() > 0 and (f2_diff > 0).sum() >= 12)
print("F2 threshold kept as review mode:", passes)

,threshold,precision,recall,F1,F2
F1 threshold,-0.272 ± 0.070,0.834 ± 0.065,0.752 ± 0.066,0.789 ± 0.057,0.766 ± 0.061
F2 threshold,-0.498 ± 0.060,0.616 ± 0.089,0.853 ± 0.070,0.710 ± 0.064,0.788 ± 0.055


,mean difference,std,folds higher
metric,,,
precision,-0.218,0.076,0/15
recall,0.101,0.072,15/15
F1,-0.079,0.051,0/15
F2,0.022,0.051,10/15


F2 threshold kept as review mode: False


## What the Two Modes Mean in Postings

Out-of-fold scores of all three repeats, each posting classified with the thresholds of its own outer fold. Counts are pooled over the repeats and divided by 3, so every development posting is counted once; the right-hand table scales them to 1,000 postings at the development fake rate.

In [4]:
thresholds = folds.set_index(["repeat", "fold"])[["threshold", "threshold_f2"]]
oof = result.oof_scores.join(thresholds, on=["repeat", "fold"])
oof["y"] = y_dev.loc[oof.index].to_numpy()
oof["profile"] = X_dev.loc[oof.index, "empty_company_profile"].map({0: "filled", 1: "empty"}).to_numpy()

def counts(frame, column):
    flag = frame["score"] >= frame[column]
    fake = frame["y"] == 1
    n_repeats = frame["repeat"].nunique()
    return pd.Series({
        "fake": fake.sum() / n_repeats,
        "detected": (fake & flag).sum() / n_repeats,
        "missed": (fake & ~flag).sum() / n_repeats,
        "false alarms": (~fake & flag).sum() / n_repeats,
        "real": (~fake).sum() / n_repeats,
    })

pooled = pd.DataFrame({"F1 threshold": counts(oof, "threshold"), "F2 threshold": counts(oof, "threshold_f2")}).T
per_1000 = pooled[["detected", "missed", "false alarms"]] * 1000 / len(dev_index)
per_1000["reviewed (flagged)"] = per_1000["detected"] + per_1000["false alarms"]
display(pd.concat({"development postings": pooled[["fake", "detected", "missed", "false alarms"]].round(0),
                   "per 1,000 postings": per_1000.round(1)}, axis=1))

by_profile = {}
for profile, part in oof.groupby("profile"):
    for mode, column in [("F1 threshold", "threshold"), ("F2 threshold", "threshold_f2")]:
        c = counts(part, column)
        by_profile[(profile, mode)] = {
            "fake": c["fake"], "recall": c["detected"] / c["fake"],
            "false alarms": c["false alarms"], "false alarms / real (%)": 100 * c["false alarms"] / c["real"],
        }
print("By company profile:")
display(pd.DataFrame(by_profile).T.round(3))

development postings                               \
                             fake detected missed false alarms   
F1 threshold                724.0    549.0  175.0        106.0   
F2 threshold                724.0    621.0  103.0        391.0   

             per 1,000 postings                                         
                       detected missed false alarms reviewed (flagged)  
F1 threshold               36.6   11.6          7.1               43.7  
F2 threshold               41.4    6.8         26.0               67.5

By company profile:


fake  recall  false alarms  false alarms / real (%)
empty  F1 threshold  484.0   0.692        94.000                    4.062
       F2 threshold  484.0   0.827       327.000                   14.131
filled F1 threshold  240.0   0.893        12.333                    0.103
       F2 threshold  240.0   0.921        63.667                    0.532

## Sensitivity to the Fake Rate

Recall and the false-alarm rate do not depend on how common fake postings are, but precision does. The development data contain about 4.8% fake postings; a real platform may see fewer. The table applies the pooled recall and false-alarm rate of each mode to other fake rates (a calculation, not a measurement).

In [5]:
rows = []
for mode, row in pooled.iterrows():
    tpr, fpr = row["detected"] / row["fake"], row["false alarms"] / row["real"]
    for rate in [y_dev.mean(), 0.02, 0.01, 0.005]:
        rows.append({"mode": mode, "fake rate (%)": round(100 * rate, 1),
                     "expected precision": tpr * rate / (tpr * rate + fpr * (1 - rate))})
pd.DataFrame(rows).pivot(index="fake rate (%)", columns="mode", values="expected precision").sort_index(ascending=False)

mode,F1 threshold,F2 threshold
fake rate (%),,
4.8,0.838,0.614
2.0,0.675,0.390
1.0,0.507,0.241
0.5,0.339,0.136


## Findings

**The F2 threshold does not pass the rule, so it is not adopted as a second operating point.** It raises recall in every fold, but the F2 gain it was chosen for is small and inconsistent.

| | F1 threshold | F2 threshold | Paired difference | Folds higher |
|---|---|---|---|---|
| Threshold | −0.272 | −0.498 | | |
| Precision | 0.834 | 0.616 | −0.218 | 0/15 |
| Recall | 0.752 | 0.853 | +0.101 | 15/15 |
| F1 | 0.789 | 0.710 | −0.079 | 0/15 |
| F2 | 0.766 | 0.788 | +0.022 | **10/15** |

- **Recall rises reliably, precision falls steeply.** Per 1,000 postings at the development fake rate (48 fake), the F2 threshold detects 41.4 instead of 36.6 fake postings and misses 6.8 instead of 11.6, but raises false alarms from 7.1 to 26.0, so the number of flagged postings grows from 44 to 68. Every second or third flagged posting is real (precision 0.62).
- **Why the F2 gain is not consistent.** Near its maximum the F2 curve is flat: many thresholds give almost the same F2 on the inner out-of-fold scores, so the chosen threshold varies between folds (standard deviation 0.060) and on the outer fold it often lands where the precision loss outweighs the recall gain. Choosing a threshold by F2 is therefore less stable than choosing it by F1.
- **The cost falls on postings without a company profile.** Recall on empty-profile postings rises from 0.69 to 0.83, but 14.1% of real empty-profile postings are flagged (4.1% at the F1 threshold). For postings with a profile, recall rises from 0.89 to 0.92 and false alarms stay rare (0.5%).
- **Precision depends on the fake rate.** With the same recall and false-alarm rate, a platform with 1% fake postings would see a precision of about 0.51 at the F1 threshold and 0.24 at the F2 threshold (calculation, not measurement).

**Conclusion.** The selected model keeps one threshold, chosen by F1. Lowering the threshold is a valid way to miss fewer fake postings, but it is a trade-off to be set by the review capacity of a platform rather than something F2 optimises reliably on these data: the recall gain of about 0.10 costs about three times as many false alarms. The test set was not used.